<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/weeks/week-02/NB02_research_methods.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 02: Research Methods in Computer Science

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-02/lab.html>.

## Overview

A proposal is only as convincing as its method. This week surveys the research methods used in computing, from controlled experiments and benchmarks to case studies, surveys, design science and formal analysis, and shows how to plan experiments with adequate power and to report them reproducibly [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to match research questions with suitable research strategies, to identify threats to validity, to compute the sample size needed for a two-group comparison, and to apply reproducibility practices when reporting computational experiments.

## The science of computing

Computer science combines mathematics, engineering and empirical science, and its methods reflect this mixture. Denning argued that computing is a science of information processes, natural and artificial, and that it follows the scientific method when it formulates and tests hypotheses [4]. Tedre and Moisseinen surveyed how the word experiment is used in computing and found several distinct meanings, from demonstrations of feasibility and trials of systems to comparisons and controlled experiments [5]. A proposal must therefore say precisely which kind of evidence it will produce.

## Choosing a research strategy

Stol and Fitzgerald organised research strategies in software engineering with three desirable qualities that no single strategy can maximise at once: generalisability over actors, precision of measurement of behaviour and realism of context [1]. Laboratory experiments offer precision but limited realism, field studies offer realism but limited control, and sample studies such as surveys offer generalisability. Formal analyses and simulations complement them. Design science, as described by Hevner and colleagues, is a further strategy for building and evaluating artefacts that solve relevant problems, with guidelines on rigour, evaluation and communication [3]. Many computing proposals combine strategies, for example a design science cycle with a controlled benchmark and a field evaluation.

## Experiments and validity

Wohlin and colleagues described the design of experiments in software engineering, from hypotheses and variables to the analysis of results, and classified threats to validity into four groups [2]. Conclusion validity concerns the statistical relation between treatment and outcome, internal validity concerns whether the relation is causal, construct validity concerns whether the measures capture the intended concepts, and external validity concerns generalisation. A frequent weakness of proposals is an experiment too small to detect the expected effect. Statistical power is the probability of detecting an effect of a given size if it exists; Figure 2.1 shows how power grows with the number of observations for small, medium and large effects. The lab computes the sample size needed for a chosen effect size, significance level and power.

![Figure 2.1](https://raw.githubusercontent.com/utkukose/rd-project-management-NB-lecture/main/weeks/week-02/figures/w02_fig1.png)

*Figure 2.1. Power of a two-sided two-group comparison at a significance level of 0.05 as a function of group size for three effect sizes, computed with a normal approximation.*

## Reproducibility

Gundersen and Kjensmo examined papers from major artificial intelligence conferences and found that few documented their methods, data and experiments completely enough for independent reproduction [6]. Pineau and colleagues reported on the reproducibility programme of the NeurIPS 2019 conference, which combined a code submission policy, a reproducibility challenge and a checklist for authors [7]. Funders increasingly expect such practices to be planned from the start. A method section that states the data, the code release, the random seeds, the statistical tests and the stopping rules is both more reproducible and more convincing to reviewers.

> **Pause and reflect.** Which of the four validity threats is the greatest risk for the evaluation you plan in your project?

# Hands-on lab

The notebook simulates a benchmark comparison of two algorithms over many datasets, applies paired tests and effect sizes, and computes the sample size needed for a planned experiment [2].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A simulated benchmark

Two algorithms are evaluated on the same 30 datasets. Because the datasets are shared, the comparison is paired.

In [ ]:
from scipy import stats
rng = np.random.default_rng(SEED)
difficulty = rng.normal(0.80, 0.06, 30)
acc_a = np.clip(difficulty + rng.normal(0.00, 0.02, 30), 0, 1)
acc_b = np.clip(difficulty + rng.normal(0.01, 0.02, 30), 0, 1)
diff = acc_b - acc_a
print(f"mean difference {diff.mean():.4f}, paired effect size dz {diff.mean() / diff.std(ddof=1):.2f}")
print("unpaired t-test p =", round(stats.ttest_ind(acc_b, acc_a).pvalue, 4))

The unpaired test ignores that both algorithms saw the same datasets, so the variation between datasets hides the difference.

### Your turn, exercise 1

Compute the p-value of the paired t-test (`scipy.stats.ttest_rel`) of `acc_b` against `acc_a`, and compare it with the Wilcoxon signed-rank test.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _paired_reference():
    return float(stats.ttest_rel(acc_b, acc_a).pvalue)

In [ ]:
p_paired = None  # your computation
check("Exercise 1", p_paired, _paired_reference())
print("Wilcoxon p =", round(stats.wilcoxon(acc_b, acc_a).pvalue, 4))

## 2. Planning the sample size

For a two-sided two-group comparison with effect size d, significance level alpha and power 1 - beta, the normal approximation gives n = 2 ((z(1 - alpha/2) + z(1 - beta)) / d)^2 per group.

### Your turn, exercise 2

Compute the required size per group, rounded up, for d = 0.5, alpha = 0.05 and power 0.8. Then check the result by simulation with the function below.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_reference():
    return int(np.ceil(2 * ((stats.norm.ppf(0.975) + stats.norm.ppf(0.8)) / 0.5) ** 2))

In [ ]:
n_per_group = None  # your computation
check("Exercise 2", n_per_group, _n_reference())

def simulated_power(n, d=0.5, reps=2000):
    hits = 0
    for _ in range(reps):
        x, y = rng.normal(0, 1, n), rng.normal(d, 1, n)
        hits += stats.ttest_ind(y, x).pvalue < 0.05
    return hits / reps

print("simulated power at the planned size:", simulated_power(_n_reference()))

## Visual exploration: How large must an experiment be?

The cell estimates the power of a two-sample t-test by simulation for three effect sizes and five sample sizes. Compare the curves with the conventional target of 0.8.

In [ ]:
from scipy.stats import ttest_ind
rng = np.random.default_rng(SEED)
ns = [10, 20, 40, 80, 160]
for d in [0.2, 0.5, 0.8]:
    power = [np.mean([ttest_ind(rng.normal(d, 1, n), rng.normal(0, 1, n)).pvalue < 0.05 for _ in range(400)]) for n in ns]
    plt.plot(ns, power, "o-", label=f"effect size d = {d}")
plt.axhline(0.8, color="k", ls="--", lw=0.8); plt.xscale("log")
plt.xlabel("sample size per group"); plt.ylabel("simulated power"); plt.title("How many runs or participants does an experiment need?"); plt.legend(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-02/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which strategy fits your project's main question, and what do you give up by choosing it?
2. Describe one threat to validity in a paper you know well and how the authors could have addressed it.
3. What would a reviewer need in order to reproduce your planned experiments?

## Weekly task and submission

Write the method section of your project idea, about 600 words: research questions, strategy, design, sample size with a power calculation, the main threats to validity and the reproducibility plan. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Research methods in a leading computing venue.** Sample twenty papers from one recent volume of a leading computing conference or journal and classify their research strategies with the framework of Stol and Fitzgerald [1]. Discuss which strategies dominate and which qualities of evidence are neglected.

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Stol, K.-J., & Fitzgerald, B. (2018). The ABC of software engineering research. *ACM Transactions on Software Engineering and Methodology*, 27(3), 11. <https://doi.org/10.1145/3241743>

[2] Wohlin, C., Runeson, P., Höst, M., Ohlsson, M. C., Regnell, B., & Wesslén, A. (2012). *Experimentation in Software Engineering*. Springer. <https://doi.org/10.1007/978-3-642-29044-2>

[3] Hevner, A. R., March, S. T., Park, J., & Ram, S. (2004). Design science in information systems research. *MIS Quarterly*, 28(1), 75-105. <https://doi.org/10.2307/25148625>

[4] Denning, P. J. (2005). Is computer science science?. *Communications of the ACM*, 48(4), 27-31. <https://doi.org/10.1145/1053291.1053309>

[5] Tedre, M., & Moisseinen, N. (2014). Experiments in computing: A survey. *The Scientific World Journal*, 2014, 549398. <https://doi.org/10.1155/2014/549398>

[6] Gundersen, O. E., & Kjensmo, S. (2018). State of the art: Reproducibility in artificial intelligence. In *Proceedings of the AAAI Conference on Artificial Intelligence*. <https://doi.org/10.1609/aaai.v32i1.11503>

[7] Pineau, J., Vincent-Lamarre, P., Sinha, K., Larivière, V., Beygelzimer, A., d'Alché-Buc, F., Fox, E., & Larochelle, H. (2021). Improving reproducibility in machine learning research (A report from the NeurIPS 2019 reproducibility program). *Journal of Machine Learning Research*, 22(164), 1-20.